```sql
Goal:
- add a first set of history features to avoid leakage
- compare against the same train - validation split

Rule:
- history features must use only past records and should not change if re-fetched at different intervals
```

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier

#### 1. Load the datasets

In [3]:
train_df=pd.read_pickle('../data/interim/train_joined_split.pkl')

In [4]:
valid_df=pd.read_pickle('../data/interim/valid_joined_split.pkl')

In [5]:
train_df.shape, valid_df.shape

((472432, 434), (118108, 434))

In [6]:
train_df = train_df.sort_values("TransactionDT").reset_index(drop=True)
valid_df = valid_df.sort_values("TransactionDT").reset_index(drop=True)

In [7]:
base_features=["TransactionAmt",
               "ProductCD",
               "card1", "card2", "card3", "card4", "card5", "card6",
               "addr1", "addr2",
               "P_emaildomain", "R_emaildomain",
               "DeviceType", "DeviceInfo"
              ]
target_col="isFraud"

In [8]:
available_features = [c for c in base_features if c in train_df.columns]

In [9]:
# Adding some transformation on features
for df_ in [train_df,valid_df]:
    df_["log_txnamt"]=np.log1p(df_["TransactionAmt"]) #log1p works best for skewed and zero values well
    df_["has_identity"]=(df_[["DeviceType", "DeviceInfo"]].notna().any(axis=1).astype(int)) #Any one value is available

In [10]:
new_features = ["log_txnamt", "has_identity"]

In [11]:
feature_cols=available_features+new_features

#### Model Training and Result Function

In [12]:
def model_input_data(train_df,valid_df,feature_cols,target_col="isFraud"):

    #Splitting Target vs Predictors
    X_train=train_df[feature_cols].copy()
    X_valid=valid_df[feature_cols].copy()

    y_train=train_df[target_col].copy()
    y_valid=valid_df[target_col].copy()

    #Seperating Categorical and Numerical Columns
    cat_cols=X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    num_cols=X_train.select_dtypes(include="number").columns.tolist()

    #Filling Missing Values
    for col in num_cols:
        X_train[col]=X_train[col].fillna(X_train[col].median())
        X_valid[col]=X_valid[col].fillna(X_valid[col].median())

    for col in cat_cols:
        X_train[col]=X_train[col].fillna("Missing").astype(str)
        X_valid[col]=X_valid[col].fillna("Missing").astype(str)
    
    # Typecasting all strings into category
    for col in cat_cols:
        X_train[col]=X_train[col].astype("category")
        X_valid[col]=X_valid[col].astype("category")
    
    return X_train, X_valid, y_train, y_valid, cat_cols

In [13]:
def model_results(train_df,valid_df,feature_cols,model_name="lgbm"):
    X_train, X_valid, y_train, y_valid, cat_cols=model_input_data(train_df,valid_df,feature_cols)
    model=LGBMClassifier(n_estimators=300,learning_rate=0.05,num_leaves=64,subsample=0.8,
                         colsample_bytree=0.8,random_state=42,class_weight="balanced")
    model.fit(X_train, y_train, categorical_feature=cat_cols)

    pred=model.predict_proba(X_valid)[:,1]

    results = {
        "n_features": len(feature_cols),
        "roc_auc": roc_auc_score(y_valid,pred),
        "pr_auc":average_precision_score(y_valid,pred),
        "pred":pred,
        "model":model
    }

    return results

#### Baseline Results

In [14]:
baseline_result = model_results(train_df, valid_df, feature_cols)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031249 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1808
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [15]:
print("Baseline ROC-AUC:", round(baseline_result["roc_auc"], 5))
print("Baseline PR-AUC :", round(baseline_result["pr_auc"], 5))

Baseline ROC-AUC: 0.80776
Baseline PR-AUC : 0.23062


#### Adding first history feature

In [18]:
train_df.groupby(["card1"])

In [ ]:
def add_history_feature(df, entity_col, amt_col="TransactionAmt", time_col="TransactionDT", prefix="card1"):
    df=df.copy()
    
    #Sort data by time
    df=df.sort_values(time_col)

    #Columns to create: 
    # 1. count of txns by before current txn
    # 2. average amount of txn before current txn
    # 3. Time since last txn

    df["cnt_txn_hist"]=0
    df["avg_txn_amt_hist"]=0
    df["time_since_last_txn_hist"]=0

    for entity, group in df.groupby(entity_col): #This gives you entity and respective group for each card1
        past_amounts=[] #store all previous txns
        last_time=None

        for i in group.index:
            df.loc[i, "txn_count_past"] = len(past_amounts) #it should be zero for first index in each group

            # average of past amounts
            if past_amounts:
                

    

In [ ]:
train_h = add_history_features(train_df, "card1", prefix="card1")
valid_h = add_history_features(valid_df, "card1", prefix="card1")

history_cols = ["card1_txn_count_past","card1_avg_amt_past","card1_secs_since_last"]
history_cols